# User Search Strategies: Satisfice vs Optimize

> **Task constraint note:** AdSERP is a forced-click task — participants must click a result. This inflates optimization behavior: some "optimizers" here might be satisficers who would reformulate in real browsing. The *relative* differences between users are still meaningful; the *absolute* rates are upper bounds.

Participants vary in how thoroughly they evaluate SERPs before clicking. Some click within the first viewport after minimal scanning; others scroll the full page, scroll back up, and compare. We segment users by their **regression rate** — the fraction of trials where they scrolled back up at least once — as a proxy for the satisfice/optimize tradeoff.

This connects to Herbert Simon's bounded rationality framework: satisficing means selecting the first option that meets a threshold; optimizing means exhaustively comparing options to find the best.

In [ ]:
# Shared data loading — see data_loader.py for all utilities
from data_loader import *
setup_plotting()
import os
import os
import csv
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


## Extract per-trial metrics

### Key Measures

| Measure | Definition | Units | Interpretation |
|---------|-----------|-------|----------------|
| Regression rate tercile | Participant classification by fraction of trials with regressions: satisficer (<43%), mixed (43-73%), optimizer (>73%) | group | Segments users by exploration strategy |
| First-viewport click rate | Fraction of trials where the user clicks without scrolling | % | Satisficers: 37%, Mixed: 15%, Optimizers: 3% |
| Optimization ROI | Additional click depth gained per unit of extra time invested | positions/s | Diminishing returns: optimizers invest 1.8x time for ~1 position deeper clicks |


In [ ]:
def get_trial_meta(trial_id):
    try:
        tree = ET.parse(os.path.join(METADATA_DIR, f'{trial_id}.xml'))
        doc_h = int(tree.find('.//document').text.split('x')[1])
        scr_h = int(tree.find('.//screen').text.split('x')[1])
        return doc_h, scr_h
    except:
        return None, None

def extract_trial(trial_id):
    path = os.path.join(MOUSE_DIR, f'{trial_id}.csv')
    scrolls, first_event_t, click_t, click_y = [], None, None, None

    with open(path) as f:
        for r in csv.DictReader(f):
            t = int(float(r['timestamp']))
            if first_event_t is None:
                first_event_t = t
            if r['event'] == 'scroll':
                scrolls.append((t, float(r['ypos'])))
            if r['event'] == 'click':
                click_t = t
                click_y = float(r['ypos'])

    if not click_t or not first_event_t:
        return None

    doc_h, scr_h = get_trial_meta(trial_id)
    if not doc_h or not scr_h:
        return None

    # Scroll gesture segmentation (200ms gap threshold)
    gestures = []
    if scrolls:
        gesture = [scrolls[0]]
        for i in range(1, len(scrolls)):
            if scrolls[i][0] - scrolls[i-1][0] > 200:
                gestures.append(gesture)
                gesture = [scrolls[i]]
            else:
                gesture.append(scrolls[i])
        gestures.append(gesture)

    n_regressions = sum(1 for g in gestures if len(g) >= 2 and g[-1][1] - g[0][1] < -10)
    max_scroll = max((s[1] for s in scrolls), default=0)
    trial_dur = (click_t - first_event_t) / 1000.0

    # Click position
    scroll_at_click = 0
    for st, sy in scrolls:
        if st <= click_t:
            scroll_at_click = sy
    # Coordinate-safe: click_y is already page-space.
    click_page_y = click_y
    n_res_approx = max(round((doc_h - 400) / 150), 1)
    per_res = (doc_h - 400) / max(n_res_approx, 1)
    click_pos = max(0, min(int((click_page_y - 200) / per_res), n_res_approx - 1))

    first_vp_click = max_scroll < 50

    # Fixation stats
    fp = os.path.join(FIXATION_DIR, f'{trial_id}.csv')
    n_fixations, total_fix_dur = 0, 0
    if os.path.exists(fp):
        with open(fp) as f:
            for row in csv.DictReader(f):
                try:
                    n_fixations += 1
                    total_fix_dur += float(row['FPOGD'])
                except:
                    pass

    return {
        'trial': trial_id,
        'pid': trial_id.split('-')[0],
        'n_regressions': n_regressions,
        'has_regression': n_regressions > 0,
        'max_scroll': max_scroll,
        'trial_dur_s': trial_dur,
        'click_pos': click_pos,
        'first_vp_click': first_vp_click,
        'n_fixations': n_fixations,
        'total_fix_dur_ms': total_fix_dur,
    }

print("Extracting trial data...")
trials = []
for i, tid in enumerate(trial_ids):
    if (i+1) % 500 == 0:
        print(f"  {i+1}/{len(trial_ids)}...")
    r = extract_trial(tid)
    if r:
        trials.append(r)

print(f"Trials: {len(trials)}")
print(f"Participants: {len(set(t['pid'] for t in trials))}")

## Aggregate by participant

In [ ]:
by_pid = defaultdict(list)
for t in trials:
    by_pid[t['pid']].append(t)

users = []
for pid, ts in by_pid.items():
    n = len(ts)
    users.append({
        'pid': pid,
        'n_trials': n,
        'regression_rate': sum(1 for t in ts if t['has_regression']) / n,
        'mean_regressions': np.mean([t['n_regressions'] for t in ts]),
        'mean_click_pos': np.mean([t['click_pos'] for t in ts]),
        'mean_trial_dur': np.mean([t['trial_dur_s'] for t in ts]),
        'median_trial_dur': np.median([t['trial_dur_s'] for t in ts]),
        'fv_click_rate': sum(1 for t in ts if t['first_vp_click']) / n,
        'mean_max_scroll': np.mean([t['max_scroll'] for t in ts]),
        'mean_fixations': np.mean([t['n_fixations'] for t in ts]),
        'mean_fix_dur': np.mean([t['total_fix_dur_ms'] for t in ts]),
    })

users.sort(key=lambda u: u['regression_rate'])
rates = [u['regression_rate'] for u in users]

print(f"Regression rate across {len(users)} participants:")
print(f"  Mean: {np.mean(rates):.1%}, Median: {np.median(rates):.1%}")
print(f"  Range: {min(rates):.0%} – {max(rates):.0%}, SD: {np.std(rates):.1%}")

## Tercile segmentation

Split participants into three groups by regression rate. The boundaries emerge from the data — we're not imposing a threshold.

In [ ]:
t1 = np.percentile(rates, 33)
t2 = np.percentile(rates, 67)

satisficers = [u for u in users if u['regression_rate'] <= t1]
mixed = [u for u in users if t1 < u['regression_rate'] <= t2]
optimizers = [u for u in users if u['regression_rate'] > t2]

print(f"Tercile boundaries: {t1:.0%}, {t2:.0%}")
print(f"Satisficers (≤{t1:.0%}): n={len(satisficers)}")
print(f"Mixed ({t1:.0%}–{t2:.0%}): n={len(mixed)}")
print(f"Optimizers (>{t2:.0%}): n={len(optimizers)}")

def profile(group):
    return {
        'Regression rate': f"{np.mean([u['regression_rate'] for u in group]):.0%}",
        'Click position': f"{np.mean([u['mean_click_pos'] for u in group]):.1f}",
        'Trial duration': f"{np.mean([u['mean_trial_dur'] for u in group]):.1f}s",
        'FV click rate': f"{np.mean([u['fv_click_rate'] for u in group]):.0%}",
        'Max scroll (px)': f"{np.mean([u['mean_max_scroll'] for u in group]):.0f}",
        'Fixations/trial': f"{np.mean([u['mean_fixations'] for u in group]):.0f}",
        'Fix time/trial': f"{np.mean([u['mean_fix_dur'] for u in group])/1000:.1f}s",
    }

print(f"\n{'Metric':<20} {'Satisficers':>12} {'Mixed':>12} {'Optimizers':>12}")
print(f"{'-'*56}")
for key in profile(satisficers):
    print(f"{key:<20} {profile(satisficers)[key]:>12} {profile(mixed)[key]:>12} {profile(optimizers)[key]:>12}")

In [ ]:
# Per-user bar chart sorted by regression rate
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

pids = [u['pid'] for u in users]
x = np.arange(len(users))

# Color by group
colors = []
for u in users:
    if u['regression_rate'] <= t1:
        colors.append('#16a34a')  # green = satisficer
    elif u['regression_rate'] <= t2:
        colors.append('#f59e0b')  # amber = mixed
    else:
        colors.append('#dc2626')  # red = optimizer

# Panel 1: Regression rate
ax = axes[0]
ax.bar(x, [u['regression_rate'] * 100 for u in users], color=colors, alpha=0.8)
ax.set_ylabel('Regression rate (%)')
ax.set_title('Per-Participant Regression Rate (sorted)')
ax.axhline(y=t1*100, color='#16a34a', linestyle='--', alpha=0.5, label=f'Satisfice ≤{t1:.0%}')
ax.axhline(y=t2*100, color='#dc2626', linestyle='--', alpha=0.5, label=f'Optimize >{t2:.0%}')
ax.legend(loc='upper left')

# Panel 2: Mean trial duration
ax = axes[1]
ax.bar(x, [u['mean_trial_dur'] for u in users], color=colors, alpha=0.8)
ax.set_ylabel('Mean trial duration (s)')
ax.set_title('Decision Time')

# Panel 3: First-viewport click rate
ax = axes[2]
ax.bar(x, [u['fv_click_rate'] * 100 for u in users], color=colors, alpha=0.8)
ax.set_ylabel('First-VP click rate (%)')
ax.set_title('First-Viewport Click Rate')
ax.set_xticks(x)
ax.set_xticklabels(pids, rotation=90, fontsize=7)
ax.set_xlabel('Participant')

plt.tight_layout()
plt.savefig('../plots-v1/plot_strategies_per_user.png', dpi=200, bbox_inches='tight')
plt.show()

## Correlation structure

How do the strategy dimensions relate to each other?

In [ ]:
# Scatter matrix of key metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = [
    ('regression_rate', 'Regression rate'),
    ('mean_trial_dur', 'Trial duration (s)'),
    ('fv_click_rate', 'FV click rate'),
    ('mean_max_scroll', 'Max scroll (px)'),
]

pairs = [(0,1), (0,2), (0,3), (1,2)]
for idx, (i, j) in enumerate(pairs):
    ax = axes[idx // 2][idx % 2]
    ki, li = metrics[i]
    kj, lj = metrics[j]
    xs = [u[ki] for u in users]
    ys = [u[kj] for u in users]
    
    for u in users:
        c = '#16a34a' if u['regression_rate'] <= t1 else '#dc2626' if u['regression_rate'] > t2 else '#f59e0b'
        ax.scatter(u[ki], u[kj], color=c, s=40, alpha=0.7, edgecolor='white', linewidth=0.5)
    
    r, p = stats.pearsonr(xs, ys)
    ax.set_xlabel(li)
    ax.set_ylabel(lj)
    ax.set_title(f'r = {r:.2f}, p = {p:.3f}')

plt.suptitle('Strategy Dimensions: Correlation Structure', fontsize=13)
plt.tight_layout()
plt.savefig('../plots-v1/plot_strategies_correlations.png', dpi=200, bbox_inches='tight')
plt.show()

## Within-user consistency

Is a user's strategy stable across trials, or do they switch between satisficing and optimizing depending on the query?

In [ ]:
# For each user: distribution of regressions per trial
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, group, label, color in [
    (axes[0], satisficers, 'Satisficers', '#16a34a'),
    (axes[1], mixed, 'Mixed', '#f59e0b'),
    (axes[2], optimizers, 'Optimizers', '#dc2626'),
]:
    # Aggregate trial-level regression counts for this group
    all_regs = []
    for u in group:
        pid_trials = [t for t in trials if t['pid'] == u['pid']]
        all_regs.extend([t['n_regressions'] for t in pid_trials])
    
    ax.hist(all_regs, bins=range(0, 10), color=color, alpha=0.7, edgecolor='white', density=True)
    ax.set_ylabel('Density')
    ax.set_title(f'{label} (n={len(group)} users, {len(all_regs)} trials)')
    zero_pct = sum(1 for r in all_regs if r == 0) / len(all_regs) * 100
    ax.text(0.95, 0.85, f'{zero_pct:.0f}% zero-regression trials',
            transform=ax.transAxes, ha='right', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[2].set_xlabel('Regressions per trial')
plt.suptitle('Within-User Regression Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('../plots-v1/plot_strategies_consistency.png', dpi=200, bbox_inches='tight')
plt.show()

# Coefficient of variation for each user
print("Within-user consistency (CV of regression count):")
for label, group in [("Satisficers", satisficers), ("Mixed", mixed), ("Optimizers", optimizers)]:
    cvs = []
    for u in group:
        pid_trials = [t['n_regressions'] for t in trials if t['pid'] == u['pid']]
        if np.mean(pid_trials) > 0:
            cvs.append(np.std(pid_trials) / np.mean(pid_trials))
    if cvs:
        print(f"  {label}: mean CV = {np.mean(cvs):.2f}")

## Extreme users

The two endpoints are particularly interesting:
- **p032** (0% regressions, 100% FV clicks, 9.2s trials) — pure satisficer, possibly not engaging with the task
- **p009** (98% regressions, 5.8 avg per trial, 0% FV clicks, 42s trials) — exhaustive optimizer

These anchor points define the behavioral range in this sample.

In [ ]:
# Click position distributions for extreme users
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, pid, label, color in [
    (ax1, 'p032', 'p032 — Pure Satisficer', '#16a34a'),
    (ax2, 'p009', 'p009 — Pure Optimizer', '#dc2626'),
]:
    pid_trials = [t for t in trials if t['pid'] == pid]
    positions = [t['click_pos'] for t in pid_trials]
    ax.hist(positions, bins=range(0, 12), color=color, alpha=0.7, edgecolor='white')
    ax.set_xlabel('Click position')
    ax.set_ylabel('Count')
    ax.set_title(f'{label}\n(n={len(pid_trials)}, mean pos={np.mean(positions):.1f})')
    ax.set_xlim(-0.5, 10.5)

plt.tight_layout()
plt.savefig('../plots-v1/plot_strategies_extremes.png', dpi=200, bbox_inches='tight')
plt.show()

## Summary

**Regression rate cleanly segments search strategy.** The tercile split reveals three coherent profiles:

| | Satisficers | Mixed | Optimizers |
|---|---|---|---|
| Regression rate | ~29% | ~58% | ~83% |
| FV click rate | 37% | 15% | 3% |
| Trial duration | 17s | 21s | 30s |
| Max scroll | 457px | 821px | 1165px |

The satisfice-optimize dimension is **continuous, not binary** — the distribution of regression rates is roughly uniform across participants, not bimodal.

**Optimizers invest 1.8x the time and 2x the fixations** compared to satisficers, but click only ~1 position deeper on average (6.0 vs 5.0). The diminishing returns are stark: twice the evaluation effort yields marginal position change.

**Within-user consistency varies.** Even "satisficers" regress on ~28% of trials — they're not rigid. The strategy adapts to query difficulty, but the baseline tendency is stable enough to segment on.

**Connection to priming:** The priming effect (faster evaluation of lexically redundant results) should be stronger for optimizers, who encounter more cumulative overlap before clicking. Satisficers who click at position 2 have barely accumulated any priming signal. This is a testable interaction — notebook `serp_priming.ipynb` can be rerun with the user segmentation as a moderator.